In [1]:
from pebble import ProcessPool
from src.tools.dataloaders import load_problem_subset, load_ground_truth_solutions, load_test_cases, get_data_dir, load_working_solutions
from src.code_evaluation.executor_standard import execute_code, check
from core.utils import setup_logging
from tqdm.auto import tqdm
import logging
setup_logging('info')

logger = logging.getLogger(__name__)

already_processed = load_working_solutions()
problems = load_problem_subset("INTERVIEW", require_solutions=True)(get_data_dir() / "test", num_problems=5000)
problems = {k: v for k, v in problems.items() if k not in already_processed}
logger.info(f"Processing {len(problems)} problems")
all_solutions = load_ground_truth_solutions(problems.keys())
test_cases = load_test_cases(problems.keys())

working_solutions = {}
all_results = {}
with ProcessPool(16) as p:
    futures = {}
    first_passing_solution = None
    for problem_id, solutions in tqdm(all_solutions.items()):
        all_results[problem_id] = {}
        logger.debug(f"Evaluating {len(solutions)} solutions for problem {problem_id}")
        for i, solution in enumerate(solutions):
            if first_passing_solution:
                break
            results = []
            futures = []
            for test_case in test_cases[problem_id]:
                futures.append(p.schedule(execute_code, (solution, test_case['input']), timeout=5))
            
            for future in futures:
                try:
                    result = future.result(timeout=5)
                    results.append(result)
                except Exception as e:
                    results.append("ERROR")
            
            all_passed = True
            all_results[problem_id][i] = []
            for result, test_case in zip(results, test_cases[problem_id]):
                # print(f"input: {repr(test_case['input'])}")
                # print(f"expected: {repr(test_case['output'])}")
                # print(f"actual: {repr(result)}")
                all_results[problem_id][i].append((test_case['input'], result, test_case['output']))
                if not check(test_case['input'], result, test_case['output']):
                    all_passed = False
                    break
            
            if all_passed:
                first_passing_solution = solution
                break
        
        if first_passing_solution:
            logger.debug(f"Problem {problem_id}: Found a passing solution")
            working_solutions[problem_id] = first_passing_solution
            first_passing_solution = None
        else:
            logger.debug(f"Problem {problem_id}: No passing solution found")
            #print(all_results[problem_id])
            working_solutions[problem_id] = None

import json

# Assuming 'all_results' is the dictionary we want to save
output_file = get_data_dir() / 'working_solutions.json'

# Save the dictionary to a JSON file
merged_results = {**already_processed, **working_solutions}

with open(output_file, 'w') as f:
    json.dump(merged_results, f, indent=4)

print(f"Results saved to {output_file}")

#print(working_solutions)
#print(all_results)

2024-09-23 21:05:56 [INFO] (core.utils) Logging level set to info
2024-09-23 21:05:56 [INFO] (__main__) Processing 0 problems


0it [00:00, ?it/s]

Results saved to /home/caleb/workspace/control/data/APPS/working_solutions.json


In [2]:
import json

# Assuming 'all_results' is the dictionary we want to save
output_file = get_data_dir() / 'working_solutions.json'

# Save the dictionary to a JSON file

with open(output_file, 'w') as f:
    json.dump(working_solutions, f, indent=4)

print(f"Results saved to {output_file}")


Results saved to /home/caleb/workspace/control/data/APPS/working_solutions.json


In [4]:
import json

# Assuming 'all_results' is the dictionary we want to save
output_file = get_data_dir() / 'working_solutions.json'
output_file_old = get_data_dir() / 'working_solutions_old.json'

with open(output_file_old, 'r') as f:
    old_results = json.load(f)
with open(output_file, 'r') as f:
    new_results = json.load(f)

print(len(old_results))
print(len(new_results))


merged = {**old_results, **new_results}

# Save the dictionary to a JSON file

with open(output_file, 'w') as f:
    json.dump(merged, f, indent=4)

print(f"Results saved to {output_file}")


1500
1227
Results saved to /home/caleb/workspace/control/data/APPS/working_solutions.json


## Another data collection run

In [1]:
from src.experiments.data_collection import get_pipeline
from core.utils import setup_logging
from src.pipeline.metrics import get_cost
setup_logging('info')

strong_model = "gpt-4-0125-preview"
weak_model = "gpt-3.5-turbo-1106"

# gpt-3.5-turbo-1106
# claude-3-opus-20240229
# claude-3-sonnet-20240229
# claude-3-haiku-20240307

data_pipeline = get_pipeline(
    strong_model=strong_model,
    weak_model=weak_model,
    name='big_run_for_vivek',
    use_cache=True,
    num_problems=200,
    skip_problems=200,
    enable_decomposition=True,
#    problem_ids=['1912']
)
results = await data_pipeline.run()

get_cost(data_pipeline, strong_model, weak_model)

#results['final_results']




2024-09-25 00:34:14 [INFO] (core.utils) Logging level set to info


  0%|          | 0/220 [00:00<?, ?it/s]

2024-09-25 00:34:14 [INFO] (src.pipeline.pipeline) Starting step 0: load_data - Using cache: True
2024-09-25 00:34:14 [INFO] (src.tools.dataloaders) Skipping the first 200 problems.
2024-09-25 00:34:16 [INFO] (src.pipeline.pipeline) Finished step 0: load_data
2024-09-25 00:34:16 [INFO] (src.pipeline.pipeline) Starting step 1: query_benign_untrusted_get_solutions - Using cache: True
2024-09-25 00:34:16 [INFO] (src.pipeline.pipeline) Finished step 1: query_benign_untrusted_get_solutions
2024-09-25 00:34:16 [INFO] (src.pipeline.pipeline) Starting step 2: remove_code_comments_query_benign - Using cache: True
2024-09-25 00:34:17 [INFO] (src.pipeline.pipeline) Finished step 2: remove_code_comments_query_benign
2024-09-25 00:34:17 [INFO] (src.pipeline.pipeline) Starting step 3: get_monitor_description_query_benign - Using cache: True
2024-09-25 00:34:18 [INFO] (src.pipeline.pipeline) Finished step 3: get_monitor_description_query_benign
2024-09-25 00:34:18 [INFO] (src.pipeline.pipeline) Start

0it [00:00, ?it/s]

2024-09-25 00:34:39 [INFO] (src.pipeline.pipeline) Finished step 6: query_benign_untrusted_usefulness
2024-09-25 00:34:39 [INFO] (src.pipeline.pipeline) Starting step 7: query_benign_results - Using cache: True
2024-09-25 00:34:39 [INFO] (src.pipeline.pipeline) Finished step 7: query_benign_results
2024-09-25 00:34:39 [INFO] (src.pipeline.pipeline) Starting step 8: generate_backdoor_ideas_query_malicious - Using cache: True
2024-09-25 00:34:42 [INFO] (src.pipeline.pipeline) Finished step 8: generate_backdoor_ideas_query_malicious
2024-09-25 00:34:42 [INFO] (src.pipeline.pipeline) Starting step 9: improve_backdoor_rarity_query_malicious - Using cache: True
2024-09-25 00:34:49 [INFO] (src.pipeline.pipeline) Finished step 9: improve_backdoor_rarity_query_malicious
2024-09-25 00:34:49 [INFO] (src.pipeline.pipeline) Starting step 10: improve_backdoor_subtlety_query_malicious - Using cache: True
2024-09-25 00:34:57 [INFO] (src.pipeline.pipeline) Finished step 10: improve_backdoor_subtlety_qu

got capacities for model gpt-4-0125-preview: 2000000, 10000
consumed capacities for model gpt-4-0125-preview: 20, 1
setting cap for model gpt-4-0125-preview: 1980000.0, 9900.0


2024-09-25 00:36:49 [WARNING] (core.llm_api.openai_llm) Encountered API error: Exception Type: RateLimitError, Error Details: Rate limit reached for gpt-4-turbo-preview in organization org-rRALD2hkdlmLWNVCKk9PG5Xq on tokens per min (TPM): Limit 2000000, Used 1998207, Requested 5044. Please try again in 97ms. Visit https://platform.openai.com/account/rate-limits to learn more., Traceback: Traceback (most recent call last):
  File "/home/caleb/workspace/control/core/llm_api/openai_llm.py", line 259, in __call__
    responses = await attempt_api_call()
                ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/caleb/workspace/control/core/llm_api/openai_llm.py", line 238, in attempt_api_call
    return await asyncio.wait_for(
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/caleb/miniconda3/envs/control/lib/python3.11/asyncio/tasks.py", line 489, in wait_for
    return fut.result()
           ^^^^^^^^^^^^
  File "/home/caleb/workspace/control/core/llm_api/openai_llm.py", line 378, in _make_ap